# Chunking comparison notebook

This notebook compares two chunking strategies side-by-side on representative documents from the knowledge base:

- RecursiveCharacterTextSplitter with chunk size 800 and overlap 100
- Semantic chunking using the project helper

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from IPython.display import display, Markdown

from app.rag.loaders import load_documents_from_manifest
from app.rag.chunkers import ChunkConfig, chunk_document

In [ ]:
documents = load_documents_from_manifest()

representative_docs = {
    "policy_wording": next(doc for doc in documents if doc.doc_type == "policy_wording"),
    "regulation": next(doc for doc in documents if doc.doc_type == "regulation"),
    "memo": next(doc for doc in documents if doc.doc_type == "memo"),
}

representative_docs

: 

In [ ]:
def compare_chunking(doc):    config = ChunkConfig(chunk_size=800, chunk_overlap=100)    recursive_chunks = chunk_document(doc, config, use_semantic=False)    semantic_chunks = chunk_document(doc, config, use_semantic=True)    return recursive_chunks, semantic_chunksdef preview_text(text: str, limit: int = 220) -> str:    cleaned = " ".join(text.split())    return cleaned[:limit] + ("..." if len(cleaned) > limit else "")def build_side_by_side_table(doc, max_rows: int = 6):    recursive_chunks, semantic_chunks = compare_chunking(doc)    rows = []    for idx in range(max(len(recursive_chunks), len(semantic_chunks))):        recursive_chunk = recursive_chunks[idx] if idx < len(recursive_chunks) else None        semantic_chunk = semantic_chunks[idx] if idx < len(semantic_chunks) else None        rows.append({            "chunk_index": idx,            "recursive_preview": preview_text(recursive_chunk.text) if recursive_chunk else "",            "semantic_preview": preview_text(semantic_chunk.text) if semantic_chunk else "",        })    return pd.DataFrame(rows).head(max_rows)summary_rows = []for label, doc in representative_docs.items():    recursive_chunks, semantic_chunks = compare_chunking(doc)    summary_rows.append({        "document": label,        "source_id": doc.source_id,        "doc_type": doc.doc_type,        "recursive_chunks": len(recursive_chunks),        "semantic_chunks": len(semantic_chunks),    })summary_df = pd.DataFrame(summary_rows)summary_df

In [ ]:
for label, doc in representative_docs.items():
    display(Markdown(f"## {label}: {doc.source_id}\n\n**doc_type**: {doc.doc_type}"))
    display(build_side_by_side_table(doc))